## Number token distance

This notebook explores token distances between numeric tokens.

- Run: run all cells top-to-bottom
- Outputs: summary statistics and plots


In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from scipy.spatial.distance import pdist, squareform

def load_model(model_name, hf_token):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=False,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    torch.cuda.empty_cache()
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        quantization_config=bnb_config,
        use_auth_token=hf_token,
    )
    return model

def load_tokenizer(model_name, hf_token):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=hf_token)
    tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

def load_config(config_file="../config.json"):
    with open(config_file) as f:
        return json.load(f)

# model_name = "meta-llama/Llama-3.1-8B"
model_name = "meta-llama/Meta-Llama-3.1-70B"
config = load_config()
hf_token = config.get("hf_token", "")
model = load_model(model_name, hf_token)
tokenizer = load_tokenizer(model_name, hf_token)

# Get the language model head weights W and convert to float32
W = model.lm_head.weight.to(torch.float32)  # Shape: (vocab_size, hidden_size)

# Calculate the pseudo-inverse W+ of the weight matrix
W_plus = torch.pinverse(W)  # Shape: (hidden_size, vocab_size)

# Get token indices for numbers 1-50
token_ids = []
for i in range(1, 51):
    tokens = tokenizer.encode(str(i), add_special_tokens=False)
    token_ids.extend(tokens)
    print(f"Token IDs for {i}: {tokens}")
print("Token IDs:", token_ids)

# Extract corresponding token vectors from W+
# W_plus shape is (hidden_size, vocab_size), we need to get columns for corresponding token_ids
vectors = W_plus[:, token_ids].T  # Shape: (num_tokens, hidden_size)

# Ensure vectors are on CPU (if memory allows)
vectors = vectors.cpu().detach().numpy()

# Calculate Euclidean distances
distance_matrix = squareform(pdist(vectors, metric='euclidean'))

/home/anhnguyen/miniconda3/envs/periodic/lib/python3.10/site-packages/transformers/models/auto/auto_factory.py:471: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3.1-70B.
401 Client Error. (Request ID: Root=1-6a1f1cc6-34e965833cb7746b52ca0803;56c14f54-a819-4691-a5f5-0390d9377fb7)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3.1-70B/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-70B is restricted. You must have access to it and be authenticated to access it. Please log in.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Manually set the color mapping limits
vmin_value = 0.05
vmax_value = 0.18  # You can adjust this value as needed

# Define the step size for label display
step = 5  # Display every 5th label

# Create heatmap
plt.figure(figsize=(6, 5))  # Increase figure size
sns.heatmap(
    distance_matrix,
    cmap="viridis",
    xticklabels=range(1, len(token_ids)+1),
    yticklabels=range(1, len(token_ids)+1),
    vmin=vmin_value,
    vmax=vmax_value
)

plt.xlabel('Token', fontsize=14)
plt.ylabel('Token', fontsize=14)

# Set x-axis labels
plt.xticks(
    ticks=np.arange(0, len(token_ids), step),
    labels=np.arange(1, len(token_ids)+1, step),
    fontsize=10,
    rotation=45  # Rotate label angle
)

# Set y-axis labels
plt.yticks(
    ticks=np.arange(0, len(token_ids), step),
    labels=np.arange(1, len(token_ids)+1, step),
    fontsize=10,
    rotation=0  # Keep y-axis labels horizontal
)

# Save heatmap
plt.savefig('../Results/number_token/heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

NameError: name 'distance_matrix' is not defined

<Figure size 600x500 with 0 Axes>